# Business Entity Resolution — Full-Data Master Notebook

Amazon ML Challenge 2026. This notebook is configured to process the complete training and test files. Stage 3 algorithms remain independently runnable so only the selected blocking experiments need to be executed.

**Important:** a 'winning' score cannot be truthfully guaranteed before running the full validation/inference on the actual data. This notebook does not claim a score in advance.


In [2]:
# === CONFIG ===
# FINAL / FULL-DATA MODE
# No training/test row cap is applied by this notebook.
SAMPLE_SIZE = None
RUN_FULL_INDEX = True
INDEX_ROWS = None
RUN_FULL_TEST = True

# Run only the Stage-3 algorithm cells you want to test.
# The notebook never silently switches algorithms.
ACTIVE_BLOCKER = "tfidf_name"

VAL_FRACTION = 0.20
RANDOM_SEED = 42
MAX_NEGATIVES_PER_S1 = 20

# Blocking experiments
TOP_K_NAME = 15
TOP_K_ADDRESS = 15
TOP_K_COMBINED = 20
SORTED_WINDOW = 11
TOKEN_BLOCK_MAX_SIZE = 500

# Final matching model
MODEL = "lightgbm"
THRESHOLD = None

print("FULL-DATA MODE: True")
print("RUN_FULL_INDEX:", RUN_FULL_INDEX)
print("RUN_FULL_TEST:", RUN_FULL_TEST)
print("ACTIVE_BLOCKER:", ACTIVE_BLOCKER)
print("TOP_K_NAME:", TOP_K_NAME)
print("TOP_K_ADDRESS:", TOP_K_ADDRESS)

FULL-DATA MODE: True
RUN_FULL_INDEX: True
RUN_FULL_TEST: True
ACTIVE_BLOCKER: tfidf_name
TOP_K_NAME: 15
TOP_K_ADDRESS: 15


## STAGE 0 — SETUP

Resolve the repository and import the existing challenge loaders, normalization utilities, and official F0.5/blocking metrics.

In [3]:
from __future__ import annotations

import importlib.util
import re
import sys
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

def find_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "student_resource" / "dataset").exists() and (
            p / "code" / "business_entity_resolution" / "src"
        ).exists():
            return p
    raise FileNotFoundError(
        "Could not find repo root. Start Jupyter from amazon-ml-challenge-2026/"
    )

ROOT = find_root()
SRC_PARENT = ROOT / "code" / "business_entity_resolution"
if str(SRC_PARENT) not in sys.path:
    sys.path.insert(0, str(SRC_PARENT))

from src.evaluation.metrics import blocking_metrics, f05_macro, f05_per_entity
from src.preprocessing.loader import (
    dataset_paths,
    load_ground_truth_sample,
    load_source,
)
from src.preprocessing.normalize import (
    extract_numbers,
    normalize_address,
    normalize_country,
    normalize_name,
    tokenize,
)

STAGE_DONE = set()
STAGE_NOTES = {}

def mark(n: int, **note):
    STAGE_DONE.add(n)
    STAGE_NOTES[n] = note

def need(*ns: int):
    missing = [n for n in ns if n not in STAGE_DONE]
    if missing:
        raise RuntimeError(
            f"Run stage(s) {missing} first. Completed: {sorted(STAGE_DONE)}"
        )

def print_stage(name, inp, out, metrics, next_step):
    print("CURRENT STAGE:", name)
    print("INPUT:", inp)
    print("OUTPUT:", out)
    print("KEY METRICS:")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
    print("NEXT STEP:", next_step)

print("ROOT:", ROOT)
print("SRC:", SRC_PARENT)
mark(0, root=str(ROOT))

ROOT: C:\Users\Admin\Desktop\AMAZON ML\amazon-ml-challenge-2026
SRC: C:\Users\Admin\Desktop\AMAZON ML\amazon-ml-challenge-2026\code\business_entity_resolution


In [4]:
need(0)
print_stage(
    name="STAGE 0 — SETUP",
    inp="notebook CWD + existing src/ tree",
    out="ROOT, loaders, normalizers, F0.5 helpers",
    metrics={"repo_root": str(ROOT), "sample_size": SAMPLE_SIZE},
    next_step="STAGE 1 — load train/test sample",
)

CURRENT STAGE: STAGE 0 — SETUP
INPUT: notebook CWD + existing src/ tree
OUTPUT: ROOT, loaders, normalizers, F0.5 helpers
KEY METRICS:
  repo_root: C:\Users\Admin\Desktop\AMAZON ML\amazon-ml-challenge-2026
  sample_size: None
NEXT STEP: STAGE 1 — load train/test sample


## STAGE 1 — DATA LOADING

Loads the **complete training universe** and the complete Source-2/Source-3 index. No `nrows` sample or index cap is used.


In [5]:
need(0)

TRAIN_PATHS = dataset_paths(str(ROOT / "student_resource" / "dataset"), "train")
TEST_PATHS = dataset_paths(str(ROOT / "student_resource" / "dataset"), "test")

def load_tsv(path: str, nrows=None) -> pd.DataFrame:
    kwargs = dict(sep="\t", dtype=str, encoding="utf-8", na_filter=False)
    if nrows is not None:
        kwargs["nrows"] = nrows
    return pd.read_csv(path, **kwargs)

s2_nrows = None
s3_nrows = None

s1_train = load_tsv(TRAIN_PATHS["source1"], nrows=None)
s2_train = load_tsv(TRAIN_PATHS["source2"], nrows=s2_nrows)
s3_train = load_tsv(TRAIN_PATHS["source3"], nrows=s3_nrows)

truth_train = load_ground_truth_sample(
    TRAIN_PATHS["ground_truth"],
    set(s1_train["entity_id"])
)

print("S1 training rows:", len(s1_train))
print("S2 indexed rows:", len(s2_train))
print("S3 indexed rows:", len(s3_train))
print("GT entries:", len(truth_train))
if len(truth_train) != len(s1_train):
    raise RuntimeError("Ground truth does not cover every loaded training Source-1 entity.")
print("Singletons:", sum(1 for s1 in s1_train["entity_id"] if not truth_train.get(s1)))
print("Columns:", list(s1_train.columns))
print("Countries:", sorted(s1_train["country"].map(normalize_country).unique().tolist()))

mark(1)

S1 training rows: 2206821
S2 indexed rows: 5034616
S3 indexed rows: 5285603
GT entries: 2206821
Singletons: 123247
Columns: ['entity_id', 'business_name', 'business_address', 'country']
Countries: ['india', 'us']


In [6]:
need(1)
print_stage(
    name="STAGE 1 — DATA LOADING",
    inp="official train TSVs",
    out="s1_train, s2_train, s3_train, truth_train",
    metrics={
        "s1_rows": len(s1_train),
        "s2_rows": len(s2_train),
        "s3_rows": len(s3_train),
        "gt_keys": len(truth_train),
        "full_s2s3_index": RUN_FULL_INDEX,
    },
    next_step="STAGE 2 — normalization",
)

CURRENT STAGE: STAGE 1 — DATA LOADING
INPUT: official train TSVs
OUTPUT: s1_train, s2_train, s3_train, truth_train
KEY METRICS:
  s1_rows: 2206821
  s2_rows: 5034616
  s3_rows: 5285603
  gt_keys: 2206821
  full_s2s3_index: True
NEXT STEP: STAGE 2 — normalization


## STAGE 2 — PREPROCESSING / NORMALIZATION

The challenge data contains spelling noise, punctuation changes, abbreviations, legal suffixes, transliteration and address-format variation. We keep the raw fields and add normalized fields.

**Important:** preprocessing is deliberately separated from blocking. You can improve this stage without changing the blocking algorithms.

In [7]:
need(1)

def add_normalized(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["name_norm"] = out["business_name"].map(normalize_name)
    out["addr_norm"] = out["business_address"].map(normalize_address)
    out["country_norm"] = out["country"].map(normalize_country)

    # Generic, country-agnostic region hint:
    # normalize_address already expands common address abbreviations.
    # We use the final address token only as an OPTIONAL blocking key.
    out["addr_last_token"] = out["addr_norm"].map(
        lambda x: x.split()[-1] if x.split() else ""
    )

    # Useful lightweight derived keys.
    out["name_first_token"] = out["name_norm"].map(
        lambda x: x.split()[0] if x.split() else ""
    )
    out["name_prefix4"] = out["name_norm"].str.replace(" ", "", regex=False).str[:4]
    out["name_prefix6"] = out["name_norm"].str.replace(" ", "", regex=False).str[:6]

    return out

s1_train = add_normalized(s1_train)
s2_train = add_normalized(s2_train)
s3_train = add_normalized(s3_train)

records_by_id = {}
for df in (s2_train, s3_train):
    for row in df.itertuples(index=False):
        d = row._asdict()
        records_by_id[d["entity_id"]] = d

print(
    s1_train[
        [
            "entity_id", "business_name", "name_norm",
            "business_address", "addr_norm",
            "country_norm", "addr_last_token"
        ]
    ].head(5).to_string(index=False)
)
print("empty name_norm rate:", float((s1_train["name_norm"] == "").mean()))
print("empty addr_norm rate:", float((s1_train["addr_norm"] == "").mean()))
print("S2/S3 lookup size:", len(records_by_id))

mark(2, n_records=len(records_by_id))

   entity_id           business_name               name_norm                                     business_address                                            addr_norm country_norm addr_last_token
S1-925783039     Orelee's Barbershop     orelee s barbershop               1795 Westchester Drive, High Point, NC     1795 westchester drive high point north carolina           us        carolina
S1-773889195             Prime Money             prime money                      17560 Ellis Road, Tahlequah, OK                  17560 ellis road tahlequah oklahoma           us        oklahoma
S1-377745466           B+ Retail Inc            b retail inc                  1712 Montebello Avenue, Phoenix, AZ               1712 montebello avenue phoenix arizona           us         arizona
S1-133037285           Christ Chapel           christ chapel    2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD 2100 cameron drive unit apartment g dundalk maryland           us        maryland
S1-755362802 Prabhav

In [8]:
need(2)
print_stage(
    name="STAGE 2 — PREPROCESSING",
    inp="raw name/address/country",
    out="normalized fields + records_by_id",
    metrics={
        "s2s3_lookup_size": len(records_by_id),
        "s1_empty_name_rate": float((s1_train["name_norm"] == "").mean()),
        "s1_empty_addr_rate": float((s1_train["addr_norm"] == "").mean()),
    },
    next_step="STAGE 3 — independently test blocking algorithms",
)

CURRENT STAGE: STAGE 2 — PREPROCESSING
INPUT: raw name/address/country
OUTPUT: normalized fields + records_by_id
KEY METRICS:
  s2s3_lookup_size: 10320219
  s1_empty_name_rate: 0.0
  s1_empty_addr_rate: 0.0
NEXT STEP: STAGE 3 — independently test blocking algorithms


# STAGE 3 — BLOCKING ALGORITHM LAB

This is the major redesign.

**Do not use one monolithic `HybridBlocker`.** Every algorithm below is independent.

Run whichever algorithm you want, inspect its measured recall/candidate count, then try another. The results are stored separately:

- `candidates_exact_name`
- `candidates_exact_address`
- `candidates_name_prefix`
- `candidates_token`
- `candidates_sorted_neighborhood`
- `candidates_tfidf_name`
- `candidates_tfidf_address`
- `candidates_tfidf_combined`
- `candidates_phonetic`

At the end, the **ACTIVATE A BLOCKER** cell lets you select one or union several algorithms for the downstream matcher.

The main research-backed candidate-generation families used here are exact blocking, multi-pass blocking, sorted neighbourhood, character n-gram TF-IDF retrieval, token blocking, and optional phonetic blocking. Record Linkage Toolkit documents blocking and sorted-neighbourhood indexing; Splink recommends multiple blocking rules whose union preserves true matches; challenge-specific 2026 repositories report strong results from country-partitioned TF-IDF top-k retrieval.

In [9]:
need(2)

def evaluate_candidates(name, candidates):
    stats = blocking_metrics(candidates, truth_train)

    counts = np.array(
        [len(candidates.get(sid, set())) for sid in s1_train["entity_id"]],
        dtype=int,
    )

    print("\n" + "=" * 70)
    print("BLOCKER:", name)
    print("=" * 70)
    print("candidate_recall:", stats.get("candidate_recall"))
    print("avg_candidates:", float(counts.mean()))
    print("median_candidates:", float(np.median(counts)))
    print("p95_candidates:", float(np.percentile(counts, 95)))
    print("p99_candidates:", float(np.percentile(counts, 99)))
    print("zero_candidate_rate:", float((counts == 0).mean()))
    print("zero_candidate_entities:", int((counts == 0).sum()))
    print("total_candidate_pairs:", int(counts.sum()))
    return stats

def empty_candidate_map():
    return {sid: set() for sid in s1_train["entity_id"]}

def inverted_index(df, key_func):
    idx = defaultdict(set)
    for row in df.itertuples(index=False):
        r = row._asdict()
        key = key_func(r)
        if key:
            idx[key].add(r["entity_id"])
    return idx

def union_maps(*maps):
    out = {sid: set() for sid in s1_train["entity_id"]}
    for mp in maps:
        for sid, ids in mp.items():
            out.setdefault(sid, set()).update(ids)
    return out

print("Blocking lab helpers loaded.")

Blocking lab helpers loaded.


### ALGORITHM 3A — Exact normalized name blocking

Fastest and highest precision candidate generator for records where normalization makes the names identical.

**Use when:** name normalization is strong.

**Limitation:** a single typo can lose the candidate completely. Therefore this should normally be one pass, not the only blocker.

In [10]:
need(2)

name_index = inverted_index(
    pd.concat([s2_train, s3_train], ignore_index=True),
    lambda r: (r["country_norm"], r["name_norm"]),
)

candidates_exact_name = {}
for r in s1_train.itertuples(index=False):
    x = r._asdict()
    candidates_exact_name[x["entity_id"]] = set(
        name_index.get((x["country_norm"], x["name_norm"]), set())
    )

evaluate_candidates("EXACT_NORMALIZED_NAME", candidates_exact_name)


BLOCKER: EXACT_NORMALIZED_NAME
candidate_recall: 0.2171091588317657
avg_candidates: 9.786513269540212
median_candidates: 1.0
p95_candidates: 60.0
p99_candidates: 176.0
zero_candidate_rate: 0.29346331215807714
zero_candidate_entities: 647621
total_candidate_pairs: 21597083


{'candidate_recall': 0.2171091588317657,
 'avg_candidates': 9.786513269540212,
 'median_candidates': 1.0,
 'p90_candidates': 15.0,
 'p95_candidates': 60.0,
 'p99_candidates': 176.0,
 'zero_candidate_rate': 0.29346331215807714}

### ALGORITHM 3B — Exact normalized address blocking

Blocks on normalized country + normalized address.

**Use when:** the address survives normalization even when the business name is noisy.

**Limitation:** addresses can be incomplete, shared, or reformatted.

In [11]:
need(2)

address_index = inverted_index(
    pd.concat([s2_train, s3_train], ignore_index=True),
    lambda r: (r["country_norm"], r["addr_norm"]),
)

candidates_exact_address = {}
for r in s1_train.itertuples(index=False):
    x = r._asdict()
    candidates_exact_address[x["entity_id"]] = set(
        address_index.get((x["country_norm"], x["addr_norm"]), set())
    )

evaluate_candidates("EXACT_NORMALIZED_ADDRESS", candidates_exact_address)


BLOCKER: EXACT_NORMALIZED_ADDRESS
candidate_recall: 0.21041990007023753
avg_candidates: 0.8948066018947617
median_candidates: 0.0
p95_candidates: 3.0
p99_candidates: 7.0
zero_candidate_rate: 0.5675675553205267
zero_candidate_entities: 1252520
total_candidate_pairs: 1974678


{'candidate_recall': 0.21041990007023753,
 'avg_candidates': 0.8948066018947617,
 'median_candidates': 0.0,
 'p90_candidates': 3.0,
 'p95_candidates': 3.0,
 'p99_candidates': 7.0,
 'zero_candidate_rate': 0.5675675553205267}

### ALGORITHM 3C — Name prefix + country blocking

Uses the first characters of the normalized business name.

This is intentionally looser than exact-name blocking and is useful for small spelling errors where the beginning of the name remains stable.

In [ ]:
need(2)

prefix_index = inverted_index(
    pd.concat([s2_train, s3_train], ignore_index=True),
    lambda r: (r["country_norm"], r["name_prefix6"]),
)

candidates_name_prefix = {}
for r in s1_train.itertuples(index=False):
    x = r._asdict()
    candidates_name_prefix[x["entity_id"]] = set(
        prefix_index.get((x["country_norm"], x["name_prefix6"]), set())
    )

evaluate_candidates("NAME_PREFIX6", candidates_name_prefix)

### ALGORITHM 3D — Token blocking

Each significant business-name token becomes a blocking key. Common tokens are capped/skipped so generic words do not explode the candidate set.

This is a standard blocking idea: multiple blocking predicates can be unioned so a record can survive through any useful key.

In [ ]:
need(2)

STOP_TOKENS = {
    "the", "and", "of", "inc", "llc", "ltd", "limited",
    "corp", "corporation", "company", "co", "pvt", "private",
    "store", "shop", "services", "service", "group"
}

all_s23 = pd.concat([s2_train, s3_train], ignore_index=True)

token_counts = Counter()
for r in all_s23.itertuples(index=False):
    toks = {
        t for t in tokenize(r.name_norm)
        if len(t) >= 3 and t not in STOP_TOKENS
    }
    for t in toks:
        token_counts[(r.country_norm, t)] += 1

token_index = defaultdict(set)
for r in all_s23.itertuples(index=False):
    toks = {
        t for t in tokenize(r.name_norm)
        if len(t) >= 3 and t not in STOP_TOKENS
    }
    for t in toks:
        key = (r.country_norm, t)
        if token_counts[key] <= TOKEN_BLOCK_MAX_SIZE:
            token_index[key].add(r.entity_id)

candidates_token = {}
for r in s1_train.itertuples(index=False):
    x = r._asdict()
    toks = {
        t for t in tokenize(x["name_norm"])
        if len(t) >= 3 and t not in STOP_TOKENS
    }
    ids = set()
    for t in toks:
        ids.update(token_index.get((x["country_norm"], t), set()))
    candidates_token[x["entity_id"]] = ids

evaluate_candidates("TOKEN_BLOCKING", candidates_token)

MemoryError: Unable to allocate 315. MiB for an array with shape (4, 10320219) and data type object

### ALGORITHM 3E — Sorted Neighbourhood on normalized name

Sort records by normalized name within country and compare records inside a fixed window.

This is specifically useful when exact blocking fails because of spelling differences. The Record Linkage Toolkit describes sorted-neighbourhood indexing as a way to recover pairs missed by exact blocking; larger windows increase candidate count.

In [ ]:
need(2)

def sorted_neighborhood_candidates(window=11):
    if window < 3 or window % 2 == 0:
        raise ValueError("SORTED_WINDOW must be an odd integer >= 3")

    s23 = pd.concat([s2_train, s3_train], ignore_index=True)
    out = empty_candidate_map()
    half = window // 2

    for country, g in pd.concat(
        [
            s1_train.assign(_side="s1"),
            s23.assign(_side="s23"),
        ],
        ignore_index=True,
    ).groupby("country_norm", sort=False):

        g = g.sort_values(["name_norm", "entity_id"], kind="mergesort").reset_index(drop=True)

        s1_positions = np.flatnonzero(g["_side"].to_numpy() == "s1")
        ids = g["entity_id"].to_numpy()

        for pos in s1_positions:
            sid = ids[pos]
            lo = max(0, pos - half)
            hi = min(len(g), pos + half + 1)
            out[sid].update(
                x for x in ids[lo:hi]
                if str(x).startswith(("S2-", "S3-"))
            )

    return out

candidates_sorted_neighborhood = sorted_neighborhood_candidates(SORTED_WINDOW)
evaluate_candidates(
    f"SORTED_NEIGHBORHOOD_WINDOW_{SORTED_WINDOW}",
    candidates_sorted_neighborhood,
)

NameError: name 'need' is not defined

### ALGORITHM 3F — Character TF-IDF nearest-neighbour blocking on business name

**This is the main algorithm to test next.**

Character 3–4 gram TF-IDF is well suited to business names with punctuation changes, abbreviations and small spelling/OCR errors. 
This cell is intentionally independent. Change `TOP_K_NAME` and rerun it.

In [ ]:
need(2)

def tfidf_topk_block(
    query_df,
    index_df,
    text_col,
    k,
    analyzer="char",
    ngram_range=(3, 4),
    min_df=1,
):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.neighbors import NearestNeighbors

    result = {sid: set() for sid in query_df["entity_id"]}

    # Country partitioning keeps unrelated records out and controls memory.
    for country in sorted(query_df["country_norm"].unique()):
        q = query_df[query_df["country_norm"] == country]
        b = index_df[index_df["country_norm"] == country]

        if q.empty or b.empty:
            continue

        texts_b = b[text_col].fillna("").astype(str).tolist()
        texts_q = q[text_col].fillna("").astype(str).tolist()

        valid_b = [i for i, t in enumerate(texts_b) if t]
        valid_q = [i for i, t in enumerate(texts_q) if t]

        if not valid_b or not valid_q:
            continue

        b_valid = [texts_b[i] for i in valid_b]
        q_valid = [texts_q[i] for i in valid_q]

        vectorizer = TfidfVectorizer(
            analyzer=analyzer,
            ngram_range=ngram_range,
            min_df=min_df,
            sublinear_tf=True,
            dtype=np.float32,
        )

        Xb = vectorizer.fit_transform(b_valid)
        Xq = vectorizer.transform(q_valid)

        k_eff = min(k, Xb.shape[0])
        nn = NearestNeighbors(
            n_neighbors=k_eff,
            metric="cosine",
            algorithm="brute",
            n_jobs=-1,
        )
        nn.fit(Xb)
        _, inds = nn.kneighbors(Xq)

        b_ids = b.iloc[valid_b]["entity_id"].to_numpy()

        for q_local, b_local_indices in zip(valid_q, inds):
            sid = q.iloc[q_local]["entity_id"]
            result[sid].update(
                str(b_ids[j]) for j in b_local_indices
            )

    return result

s23 = pd.concat([s2_train, s3_train], ignore_index=True)

candidates_tfidf_name = tfidf_topk_block(
    s1_train,
    s23,
    text_col="name_norm",
    k=TOP_K_NAME,
    analyzer="char",
    ngram_range=(3, 4),
)

evaluate_candidates(
    f"TFIDF_CHAR_NAME_TOP_{TOP_K_NAME}",
    candidates_tfidf_name,
)

### ALGORITHM 3G — Character TF-IDF nearest-neighbour blocking on address

Address noise is often structurally different from name noise. This pass is deliberately separate so we can measure whether it adds true matches that name retrieval misses.

The final union can combine the two without forcing the same retrieval strategy on both fields.

In [ ]:
need(2)

candidates_tfidf_address = tfidf_topk_block(
    s1_train,
    s23,
    text_col="addr_norm",
    k=TOP_K_ADDRESS,
    analyzer="char",
    ngram_range=(3, 5),
)

evaluate_candidates(
    f"TFIDF_CHAR_ADDRESS_TOP_{TOP_K_ADDRESS}",
    candidates_tfidf_address,
)

### ALGORITHM 3H — Combined name + address TF-IDF retrieval

A single retrieval channel using both fields.

This is an experiment, not automatically the final choice. Compare its recall and candidate count with the separate name/address channels.

In [ ]:
need(2)

s1_combo = s1_train.copy()
s23_combo = s23.copy()

s1_combo["combo_text"] = (
    s1_combo["name_norm"].fillna("") + " " +
    s1_combo["addr_norm"].fillna("")
).str.strip()

s23_combo["combo_text"] = (
    s23_combo["name_norm"].fillna("") + " " +
    s23_combo["addr_norm"].fillna("")
).str.strip()

candidates_tfidf_combined = tfidf_topk_block(
    s1_combo,
    s23_combo,
    text_col="combo_text",
    k=TOP_K_COMBINED,
    analyzer="char",
    ngram_range=(3, 4),
)

evaluate_candidates(
    f"TFIDF_CHAR_COMBINED_TOP_{TOP_K_COMBINED}",
    candidates_tfidf_combined,
)

### ALGORITHM 3I — Optional phonetic blocking

Useful for names with phonetic/spelling variation. This is an **optional** experiment because phonetic encodings can collide heavily, especially for short names and multilingual data.

If `jellyfish` is unavailable, this cell tells you instead of silently changing the algorithm.

In [ ]:
need(2)

try:
    import jellyfish
except ImportError:
    jellyfish = None

if jellyfish is None:
    print("jellyfish is not installed.")
    print("Install it only if you want to test this algorithm: pip install jellyfish")
    candidates_phonetic = empty_candidate_map()
else:
    def phonetic_key(name):
        toks = [
            t for t in name.split()
            if len(t) >= 3 and t not in STOP_TOKENS
        ]
        if not toks:
            return ""
        return " ".join(jellyfish.metaphone(t) for t in toks[:3])

    phon_index = inverted_index(
        s23,
        lambda r: (r["country_norm"], phonetic_key(r["name_norm"])),
    )

    candidates_phonetic = {}
    for r in s1_train.itertuples(index=False):
        x = r._asdict()
        candidates_phonetic[x["entity_id"]] = set(
            phon_index.get(
                (x["country_norm"], phonetic_key(x["name_norm"])),
                set(),
            )
        )

    evaluate_candidates("PHONETIC_METAPHONE", candidates_phonetic)

## STAGE 3J — MULTI-PASS UNION EXPERIMENTS

A standard ER principle is that one blocking rule should not have to catch every true match. Multiple blocking rules can be unioned: a pair survives if **any** selected pass retrieves it.

Run this only after you have measured the individual passes.

In [ ]:
need(2)

# Example high-recall union.
# Change this list after seeing your measured Stage-3 results.
SELECTED_BLOCKING_PASSES = [
    "tfidf_name",
    "tfidf_address",
]

BLOCKER_RESULTS = {
    "exact_name": candidates_exact_name,
    "exact_address": candidates_exact_address,
    "name_prefix": candidates_name_prefix,
    "token": candidates_token,
    "sorted_neighborhood": candidates_sorted_neighborhood,
    "tfidf_name": candidates_tfidf_name,
    "tfidf_address": candidates_tfidf_address,
    "tfidf_combined": candidates_tfidf_combined,
    "phonetic": candidates_phonetic,
}

candidates_union = union_maps(
    *(BLOCKER_RESULTS[name] for name in SELECTED_BLOCKING_PASSES)
)

evaluate_candidates(
    "MULTI_PASS_UNION: " + " + ".join(SELECTED_BLOCKING_PASSES),
    candidates_union,
)

## STAGE 3K — ACTIVATE THE CANDIDATE SET

**This is the only Stage-3 decision cell.**

After experimenting, choose the candidate map that you want the matcher to consume.

Do not automatically choose the largest candidate set. The challenge explicitly asks for scalable candidate generation, and the candidate file is reviewed for blocking quality. Your target is **high blocking recall with a controlled candidate count**. The challenge-specific public implementations use TF-IDF top-k retrieval and multi-pass blocking rather than a single brittle exact-key rule.

In [ ]:
# Select exactly one candidate map after running the corresponding Stage-3 algorithm cell.
# Examples: "exact_name", "token", "tfidf_name", "tfidf_address", "tfidf_combined", "union"

BLOCKER_RESULTS = {}
for _name, _var in [
    ("exact_name", "candidates_exact_name"),
    ("exact_address", "candidates_exact_address"),
    ("name_prefix", "candidates_name_prefix"),
    ("token", "candidates_token"),
    ("sorted_neighborhood", "candidates_sorted_neighborhood"),
    ("tfidf_name", "candidates_tfidf_name"),
    ("tfidf_address", "candidates_tfidf_address"),
    ("tfidf_combined", "candidates_tfidf_combined"),
    ("phonetic", "candidates_phonetic"),
    ("union", "candidates_union"),
]:
    if _var in globals():
        BLOCKER_RESULTS[_name] = globals()[_var]

if ACTIVE_BLOCKER not in BLOCKER_RESULTS:
    raise RuntimeError(
        f"ACTIVE_BLOCKER={ACTIVE_BLOCKER!r} has not been run. "
        f"Run its Stage-3 algorithm cell first. Available: {sorted(BLOCKER_RESULTS)}"
    )

candidates_train = BLOCKER_RESULTS[ACTIVE_BLOCKER]
active_stats = evaluate_candidates("ACTIVE: " + ACTIVE_BLOCKER, candidates_train)
mark(3, active_blocker=ACTIVE_BLOCKER, **active_stats)


In [ ]:
need(3)
print_stage(
    name="STAGE 3 — BLOCKING / CANDIDATE GENERATION",
    inp=f"normalized S1/S2/S3 + ACTIVE_BLOCKER={ACTIVE_BLOCKER}",
    out="candidates_train",
    metrics=STAGE_NOTES[3],
    next_step="STAGE 4 — inspect blocking recall; do not train until recall is acceptable",
)

## STAGE 4 — BLOCKING EVALUATION

This stage is the **gatekeeper**.

The downstream matcher cannot recover a true match that blocking never produces. Therefore we compare:

- candidate recall
- average candidates/S1
- median
- p95 / p99
- zero-candidate rate
- total candidate pairs

Do not move to model training simply because the code runs.

In [ ]:
need(3)

block_stats = blocking_metrics(candidates_train, truth_train)

true_pairs = sum(len(t) for t in truth_train.values())
found_pairs = sum(
    len(truth & candidates_train.get(sid, set()))
    for sid, truth in truth_train.items()
)

available_true_pairs = sum(
    1
    for sid, truth in truth_train.items()
    for cid in truth
    if cid in records_by_id
)

counts = np.array(
    [len(candidates_train.get(sid, set())) for sid in s1_train["entity_id"]],
    dtype=int,
)

print("candidate_recall:", block_stats["candidate_recall"])
print("true_match_ids_in_gt:", true_pairs)
print("true_matches_retrieved:", found_pairs)
print("true_match_ids_present_in_current_index:", available_true_pairs)
print("avg_candidates:", float(counts.mean()))
print("median_candidates:", float(np.median(counts)))
print("p95_candidates:", float(np.percentile(counts, 95)))
print("p99_candidates:", float(np.percentile(counts, 99)))
print("zero_candidate_rate:", float((counts == 0).mean()))

mark(4, **block_stats)

In [ ]:
need(4)
print_stage(
    name="STAGE 4 — BLOCKING EVALUATION",
    inp=f"candidates_train from {ACTIVE_BLOCKER}",
    out="measured blocking recall/candidate-size statistics",
    metrics={
        **block_stats,
        "avg_candidates": float(np.mean([len(v) for v in candidates_train.values()])),
        "zero_candidate_rate": float(
            np.mean([len(v) == 0 for v in candidates_train.values()])
        ),
    },
    next_step="STAGE 5 — feature engineering on the chosen candidate set",
)

# STAGE 5 — PAIR FEATURE ENGINEERING

Now the matcher sees only the **active candidate set**.

The feature set below is intentionally stronger than the original notebook: name/address fuzzy similarity, token overlap, character n-gram similarity, number overlap, length/containment signals, country agreement, and candidate rank where available.

In [ ]:
need(4)

from rapidfuzz.fuzz import (
    ratio,
    WRatio,
    token_set_ratio,
    token_sort_ratio,
)

FEATURE_NAMES = [
    "name_ratio",
    "name_wratio",
    "name_token_set",
    "name_token_sort",
    "name_jaccard",
    "name_length_ratio",
    "name_contains",
    "addr_ratio",
    "addr_wratio",
    "addr_token_set",
    "addr_token_sort",
    "addr_jaccard",
    "addr_length_ratio",
    "addr_contains",
    "num_jaccard",
    "same_country",
    "same_addr_last_token",
    "same_name_prefix4",
]

def _jaccard(a, b):
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

def _length_ratio(a, b):
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return min(len(a), len(b)) / max(len(a), len(b))

def pair_features(s1, s2):
    n1, n2 = s1["name_norm"], s2["name_norm"]
    a1, a2 = s1["addr_norm"], s2["addr_norm"]
    c1, c2 = s1["country_norm"], s2["country_norm"]

    return {
        "name_ratio": ratio(n1, n2) / 100.0,
        "name_wratio": WRatio(n1, n2) / 100.0,
        "name_token_set": token_set_ratio(n1, n2) / 100.0,
        "name_token_sort": token_sort_ratio(n1, n2) / 100.0,
        "name_jaccard": _jaccard(tokenize(n1), tokenize(n2)),
        "name_length_ratio": _length_ratio(n1, n2),
        "name_contains": float(bool(n1 and n2 and (n1 in n2 or n2 in n1))),

        "addr_ratio": ratio(a1, a2) / 100.0,
        "addr_wratio": WRatio(a1, a2) / 100.0,
        "addr_token_set": token_set_ratio(a1, a2) / 100.0,
        "addr_token_sort": token_sort_ratio(a1, a2) / 100.0,
        "addr_jaccard": _jaccard(tokenize(a1), tokenize(a2)),
        "addr_length_ratio": _length_ratio(a1, a2),
        "addr_contains": float(bool(a1 and a2 and (a1 in a2 or a2 in a1))),

        "num_jaccard": _jaccard(
            extract_numbers(s1["business_address"]),
            extract_numbers(s2["business_address"]),
        ),

        "same_country": float(bool(c1) and c1 == c2),
        "same_addr_last_token": float(
            bool(s1.get("addr_last_token"))
            and s1.get("addr_last_token") == s2.get("addr_last_token")
        ),
        "same_name_prefix4": float(
            bool(s1.get("name_prefix4"))
            and s1.get("name_prefix4") == s2.get("name_prefix4")
        ),
    }

rng = np.random.default_rng(RANDOM_SEED)
rows = []
skipped_missing = 0

for rec in s1_train.itertuples(index=False):
    s1 = rec._asdict()
    sid = s1["entity_id"]
    cands = list(candidates_train.get(sid, set()))
    truth = truth_train.get(sid, set())

    pos = [c for c in cands if c in truth]
    neg = [c for c in cands if c not in truth]

    if len(neg) > MAX_NEGATIVES_PER_S1:
        pick = rng.choice(len(neg), size=MAX_NEGATIVES_PER_S1, replace=False)
        neg = [neg[i] for i in pick]

    for cid in pos + neg:
        if cid not in records_by_id:
            skipped_missing += 1
            continue

        feat = pair_features(s1, records_by_id[cid])
        feat["s1_id"] = sid
        feat["cand_id"] = cid
        feat["label"] = int(cid in truth)
        rows.append(feat)

pairs_df = pd.DataFrame(rows)

print("pair rows:", len(pairs_df))
print("skipped missing:", skipped_missing)

if len(pairs_df):
    print(pairs_df["label"].value_counts().to_string())
    print(pairs_df[FEATURE_NAMES + ["label"]].head(5).to_string(index=False))
else:
    raise RuntimeError("No training pairs. Improve blocking or increase index coverage.")

mark(5, n_pairs=len(pairs_df))

## STAGE 6 — MODEL TRAINING

LightGBM remains the primary matcher. The important split is **by S1 entity**, not by pair, to avoid leakage from the same S1 entity appearing in both train and validation.

Challenge-specific public implementations also use LightGBM with grouped validation and macro-F0.5 threshold tuning.

In [ ]:
need(5)

s1_ids = pairs_df["s1_id"].unique()
rng2 = np.random.default_rng(RANDOM_SEED)
s1_ids = s1_ids[rng2.permutation(len(s1_ids))]

n_val = max(1, int(len(s1_ids) * VAL_FRACTION))
s1_val = set(s1_ids[:n_val])
s1_fit = set(s1_ids[n_val:])

if not s1_fit:
    s1_fit, s1_val = s1_val, set()

fit_df = pairs_df[pairs_df["s1_id"].isin(s1_fit)]
val_pairs = pairs_df[pairs_df["s1_id"].isin(s1_val)]

X_fit = fit_df[FEATURE_NAMES].to_numpy(dtype=float)
y_fit = fit_df["label"].to_numpy(dtype=int)

if MODEL == "lightgbm":
    try:
        import lightgbm as lgb
        model = lgb.LGBMClassifier(
            n_estimators=400,
            num_leaves=31,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
        model_name_used = "LightGBM"
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingClassifier
        model = HistGradientBoostingClassifier(
            max_depth=7,
            learning_rate=0.04,
            max_iter=300,
            random_state=RANDOM_SEED,
        )
        model_name_used = "sklearn HistGradientBoostingClassifier"
else:
    from sklearn.ensemble import HistGradientBoostingClassifier
    model = HistGradientBoostingClassifier(
        max_depth=7,
        learning_rate=0.04,
        max_iter=300,
        random_state=RANDOM_SEED,
    )
    model_name_used = "sklearn HistGradientBoostingClassifier"

model.fit(X_fit, y_fit)

print("model:", model_name_used)
print("fit S1:", len(s1_fit))
print("validation S1:", len(s1_val))
print("fit pairs:", len(fit_df))
print("positive pairs:", int(y_fit.sum()))
print("negative pairs:", int((y_fit == 0).sum()))

mark(6, model=model_name_used)

## STAGE 7 — VALIDATION + ENTITY-LEVEL F0.5

In [ ]:
need(6)

def predict_entities(
    s1_df,
    cand_map,
    recs,
    clf,
    threshold,
):
    out = {}

    for rec in s1_df.itertuples(index=False):
        s1 = rec._asdict()
        sid = s1["entity_id"]
        kept = set()

        for cid in cand_map.get(sid, set()):
            if cid not in recs:
                continue

            feat = pair_features(s1, recs[cid])
            x = np.array(
                [feat[k] for k in FEATURE_NAMES],
                dtype=float,
            ).reshape(1, -1)

            proba = float(clf.predict_proba(x)[0, 1])
            if proba >= threshold:
                kept.add(cid)

        out[sid] = kept

    return out

working_threshold = 0.5 if THRESHOLD is None else float(THRESHOLD)

s1_val_df = s1_train[
    s1_train["entity_id"].isin(s1_val)
]

val_pred = predict_entities(
    s1_val_df,
    candidates_train,
    records_by_id,
    model,
    working_threshold,
)

val_truth = {
    sid: truth_train.get(sid, set())
    for sid in val_pred
}

val_f05 = float(f05_macro(val_pred, val_truth))

print("threshold:", working_threshold)
print("validation S1:", len(val_pred))
print("macro F0.5:", val_f05)
print("empty predictions:", sum(not v for v in val_pred.values()))

mark(7, val_f05=val_f05)

## STAGE 8 — THRESHOLD OPTIMIZATION

Because the official metric is macro F0.5 and is precision-heavy, `0.5` is not assumed to be optimal. We sweep the threshold on the held-out S1 entities.

In [ ]:
need(7)

cuts = np.round(np.linspace(0.10, 0.95, 18), 2)
rows_t = []

for t in cuts:
    pred_t = predict_entities(
        s1_val_df,
        candidates_train,
        records_by_id,
        model,
        float(t),
    )
    truth_t = {k: truth_train.get(k, set()) for k in pred_t}

    score = float(f05_macro(pred_t, truth_t))
    rows_t.append({
        "threshold": float(t),
        "macro_f05": score,
        "mean_matches": float(
            np.mean([len(v) for v in pred_t.values()])
        ),
    })

sweep_df = pd.DataFrame(rows_t)
print(sweep_df.to_string(index=False))

best_row = sweep_df.loc[sweep_df["macro_f05"].idxmax()]
swept_best = float(best_row["threshold"])
BEST_THRESHOLD = (
    float(THRESHOLD)
    if THRESHOLD is not None
    else swept_best
)

print("BEST_THRESHOLD:", BEST_THRESHOLD)
print("best validation F0.5:", float(best_row["macro_f05"]))

mark(8, BEST_THRESHOLD=BEST_THRESHOLD)

## STAGE 9 — FINAL TRAINING

Refit the matcher on all available candidate pairs after threshold selection.

In [ ]:
need(8)

X_all = pairs_df[FEATURE_NAMES].to_numpy(dtype=float)
y_all = pairs_df["label"].to_numpy(dtype=int)

if MODEL == "lightgbm":
    try:
        import lightgbm as lgb
        final_model = lgb.LGBMClassifier(
            n_estimators=400,
            num_leaves=31,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingClassifier
        final_model = HistGradientBoostingClassifier(
            max_depth=7,
            learning_rate=0.04,
            max_iter=300,
            random_state=RANDOM_SEED,
        )
else:
    from sklearn.ensemble import HistGradientBoostingClassifier
    final_model = HistGradientBoostingClassifier(
        max_depth=7,
        learning_rate=0.04,
        max_iter=300,
        random_state=RANDOM_SEED,
    )

final_model.fit(X_all, y_all)

print("final pairs:", len(pairs_df))
print("positive:", int(y_all.sum()))
print("BEST_THRESHOLD:", BEST_THRESHOLD)

mark(9)

## STAGE 10 — TEST INFERENCE

Only after the training experiments are satisfactory.

This notebook is configured for the complete test set. No sample inference is used.

In [ ]:
need(9)

test_s1_n = None
test_s23_n = None

s1_test = load_tsv(TEST_PATHS["source1"], nrows=test_s1_n)
s2_test = load_tsv(TEST_PATHS["source2"], nrows=test_s23_n)
s3_test = load_tsv(TEST_PATHS["source3"], nrows=test_s23_n)

s1_test = add_normalized(s1_test)
s2_test = add_normalized(s2_test)
s3_test = add_normalized(s3_test)

test_records = {}
for df in (s2_test, s3_test):
    for row in df.itertuples(index=False):
        d = row._asdict()
        test_records[d["entity_id"]] = d

test_s23 = pd.concat([s2_test, s3_test], ignore_index=True)

# Re-run the selected blocking algorithm on test data.
# This maps the experimental algorithm names to the same implementations.
if ACTIVE_BLOCKER == "exact_name":
    idx = inverted_index(
        test_s23,
        lambda r: (r["country_norm"], r["name_norm"]),
    )
    candidates_test = {
        r.entity_id: set(idx.get((r.country_norm, r.name_norm), set()))
        for r in s1_test.itertuples(index=False)
    }

elif ACTIVE_BLOCKER == "exact_address":
    idx = inverted_index(
        test_s23,
        lambda r: (r["country_norm"], r["addr_norm"]),
    )
    candidates_test = {
        r.entity_id: set(idx.get((r.country_norm, r.addr_norm), set()))
        for r in s1_test.itertuples(index=False)
    }

elif ACTIVE_BLOCKER == "name_prefix":
    idx = inverted_index(
        test_s23,
        lambda r: (r["country_norm"], r["name_prefix6"]),
    )
    candidates_test = {
        r.entity_id: set(idx.get((r.country_norm, r.name_prefix6), set()))
        for r in s1_test.itertuples(index=False)
    }

elif ACTIVE_BLOCKER == "token":
    # Rebuild the token index for test data.
    token_counts_test = Counter()
    for r in test_s23.itertuples(index=False):
        toks = {
            t for t in tokenize(r.name_norm)
            if len(t) >= 3 and t not in STOP_TOKENS
        }
        for t in toks:
            token_counts_test[(r.country_norm, t)] += 1

    idx = defaultdict(set)
    for r in test_s23.itertuples(index=False):
        toks = {
            t for t in tokenize(r.name_norm)
            if len(t) >= 3 and t not in STOP_TOKENS
        }
        for t in toks:
            key = (r.country_norm, t)
            if token_counts_test[key] <= TOKEN_BLOCK_MAX_SIZE:
                idx[key].add(r.entity_id)

    candidates_test = {}
    for r in s1_test.itertuples(index=False):
        toks = {
            t for t in tokenize(r.name_norm)
            if len(t) >= 3 and t not in STOP_TOKENS
        }
        ids = set()
        for t in toks:
            ids.update(idx.get((r.country_norm, t), set()))
        candidates_test[r.entity_id] = ids

elif ACTIVE_BLOCKER == "sorted_neighborhood":
    # Temporarily swap the training globals used by the helper.
    old_s1, old_s2, old_s3 = s1_train, s2_train, s3_train
    s1_train, s2_train, s3_train = s1_test, s2_test, s3_test
    candidates_test = sorted_neighborhood_candidates(SORTED_WINDOW)
    s1_train, s2_train, s3_train = old_s1, old_s2, old_s3

elif ACTIVE_BLOCKER == "tfidf_name":
    candidates_test = tfidf_topk_block(
        s1_test, test_s23, "name_norm",
        TOP_K_NAME, "char", (3, 4)
    )

elif ACTIVE_BLOCKER == "tfidf_address":
    candidates_test = tfidf_topk_block(
        s1_test, test_s23, "addr_norm",
        TOP_K_ADDRESS, "char", (3, 5)
    )

elif ACTIVE_BLOCKER == "tfidf_combined":
    q = s1_test.copy()
    b = test_s23.copy()
    q["combo_text"] = q["name_norm"] + " " + q["addr_norm"]
    b["combo_text"] = b["name_norm"] + " " + b["addr_norm"]
    candidates_test = tfidf_topk_block(
        q, b, "combo_text",
        TOP_K_COMBINED, "char", (3, 4)
    )

elif ACTIVE_BLOCKER == "union":
    # Rebuild the individual selected passes on test data.
    test_maps = []

    if "exact_name" in SELECTED_BLOCKING_PASSES:
        idx = inverted_index(
            test_s23,
            lambda r: (r["country_norm"], r["name_norm"]),
        )
        test_maps.append({
            r.entity_id: set(idx.get((r.country_norm, r.name_norm), set()))
            for r in s1_test.itertuples(index=False)
        })

    if "exact_address" in SELECTED_BLOCKING_PASSES:
        idx = inverted_index(
            test_s23,
            lambda r: (r["country_norm"], r["addr_norm"]),
        )
        test_maps.append({
            r.entity_id: set(idx.get((r.country_norm, r.addr_norm), set()))
            for r in s1_test.itertuples(index=False)
        })

    if "name_prefix" in SELECTED_BLOCKING_PASSES:
        idx = inverted_index(
            test_s23,
            lambda r: (r["country_norm"], r["name_prefix6"]),
        )
        test_maps.append({
            r.entity_id: set(idx.get((r.country_norm, r.name_prefix6), set()))
            for r in s1_test.itertuples(index=False)
        })

    if "tfidf_name" in SELECTED_BLOCKING_PASSES:
        test_maps.append(
            tfidf_topk_block(
                s1_test, test_s23, "name_norm",
                TOP_K_NAME, "char", (3, 4)
            )
        )

    if "tfidf_address" in SELECTED_BLOCKING_PASSES:
        test_maps.append(
            tfidf_topk_block(
                s1_test, test_s23, "addr_norm",
                TOP_K_ADDRESS, "char", (3, 5)
            )
        )

    if "sorted_neighborhood" in SELECTED_BLOCKING_PASSES:
        old_s1, old_s2, old_s3 = s1_train, s2_train, s3_train
        s1_train, s2_train, s3_train = s1_test, s2_test, s3_test
        test_maps.append(sorted_neighborhood_candidates(SORTED_WINDOW))
        s1_train, s2_train, s3_train = old_s1, old_s2, old_s3

    candidates_test = {}
    for sid in s1_test["entity_id"]:
        candidates_test[sid] = set()
        for mp in test_maps:
            candidates_test[sid].update(mp.get(sid, set()))

else:
    raise ValueError(f"Unsupported ACTIVE_BLOCKER for test inference: {ACTIVE_BLOCKER}")

pred_test = predict_entities(
    s1_test,
    candidates_test,
    test_records,
    final_model,
    BEST_THRESHOLD,
)

counts_test = np.array(
    [len(candidates_test[sid]) for sid in s1_test["entity_id"]]
)

matches_test = np.array(
    [len(pred_test[sid]) for sid in s1_test["entity_id"]]
)

print("ACTIVE_BLOCKER:", ACTIVE_BLOCKER)
print("FULL TEST MODE:", RUN_FULL_TEST)
print("test S1 rows:", len(s1_test))
print("avg candidates:", float(counts_test.mean()))
print("avg predicted matches:", float(matches_test.mean()))
print("zero candidates:", int((counts_test == 0).sum()))
print("zero predicted matches:", int((matches_test == 0).sum()))

mark(10)

## STAGE 11 — OUTPUT GENERATION

The challenge requires:

- `matching_results.tsv` — leaderboard-scored predictions
- `candidate_pairs.tsv` — the exact final candidate set fed into the matcher

Every predicted match must be a subset of its candidate list.

In [ ]:
need(10)

out_dir = ROOT / "output"
out_dir.mkdir(exist_ok=True)

match_path = out_dir / "matching_results.tsv"
cand_path = out_dir / "candidate_pairs.tsv"

def pack_ids(ids):
    return ",".join(sorted(ids))

match_df = pd.DataFrame([
    {
        "source1_entity_id": sid,
        "matched_entity_ids": pack_ids(pred_test[sid]),
    }
    for sid in s1_test["entity_id"]
])

cand_df = pd.DataFrame([
    {
        "source1_entity_id": sid,
        "candidate_entity_ids": pack_ids(candidates_test[sid]),
    }
    for sid in s1_test["entity_id"]
])

match_df.to_csv(match_path, sep="\t", index=False, encoding="utf-8")
cand_df.to_csv(cand_path, sep="\t", index=False, encoding="utf-8")

subset_ok = all(
    pred_test[sid] <= candidates_test[sid]
    for sid in s1_test["entity_id"]
)

print("wrote:", match_path)
print("wrote:", cand_path)
print("rows:", len(match_df))
print("matches subset of candidates:", subset_ok)

if not RUN_FULL_TEST:
    raise RuntimeError("Final notebook must run with RUN_FULL_TEST=True.")

mark(11)

## STAGE 12 — OFFICIAL VALIDATION

Run the challenge-provided validator.

The validator runs on the complete test output.

In [ ]:
need(11)

spec = importlib.util.spec_from_file_location(
    "validate_submission",
    ROOT / "student_resource" / "utils" / "validate_submission.py",
)

validate_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validate_mod)

errors, warnings = validate_mod.validate(
    str(match_path),
    str(cand_path),
    str(ROOT / "student_resource" / "dataset" / "test"),
    check_ids=True,
)

for w in warnings:
    print("WARNING:", w)

if errors:
    print("FAIL:")
    for i, e in enumerate(errors, 1):
        print(f"  {i}. {e}")
else:
    print("PASS — complete submission format validated.")

mark(12)

# FINAL SUBMISSION CHECKLIST

Before the final run:

- [ ] Stage 1 loads the complete training universe.
- [ ] Stage 2 normalization has been checked.
- [ ] Multiple Stage 3 blocking algorithms have been measured.
- [ ] Stage 4 blocking recall is strong enough.
- [ ] The final candidate set is not unnecessarily huge.
- [ ] Stage 5 features are generated only from final candidates.
- [ ] Validation split is by S1 entity.
- [ ] Threshold is selected using entity-level macro F0.5.
- [ ] Final model is retrained on all training pairs.
- [ ] `RUN_FULL_TEST = True`.
- [ ] Every test S1 gets a row.
- [ ] `matching_results.tsv` predictions are a subset of `candidate_pairs.tsv`.
- [ ] Official validator returns PASS.

### Final files

```text
output/
├── matching_results.tsv
└── candidate_pairs.tsv
```

The challenge documentation states that `matching_results.tsv` is the leaderboard-scored file and `candidate_pairs.tsv` is part of the final submission package and is reviewed for blocking quality.